In [1]:
import numpy as np
import pandas as pd

In [ ]:
insurance = pd.read_csv(r'C:insurance_claims_c.csv', parse_dates=['claim_submitted_date', 'claim_approved_date'])
billing = pd.read_csv(r'C:billing_and_costs_c.csv')
patients = pd.read_csv(r'C:patients_c.csv')

print(insurance.shape)
print(insurance.dtypes)
insurance.head()

(11599, 11)
claim_id                           str
billing_id                         str
patient_id                         str
insurer_name                       str
claim_submitted_date    datetime64[us]
claim_approved_date     datetime64[us]
processing_time_days           float64
claim_amount                   float64
approved_amount                float64
claim_status                       str
denial_reason                      str
dtype: object


,claim_id,billing_id,patient_id,insurer_name,claim_submitted_date,claim_approved_date,processing_time_days,claim_amount,approved_amount,claim_status,denial_reason
0,CLM-000001,BIL-000001,PAT-07844,Medicaid State Plan,2021-01-09,2021-02-12,34.0,2415.02,2164.82,Approved,NaN
1,CLM-000002,BIL-000002,PAT-06074,Medicare Federal,2019-07-03,2019-07-17,14.0,4268.26,3712.24,Approved,NaN
2,CLM-000003,BIL-000003,PAT-01784,Medicaid State Plan,2022-11-16,2022-12-04,18.0,500.00,0.00,Denied,Incomplete documentation
3,CLM-000004,BIL-000004,PAT-07425,BlueCross BlueShield,2017-02-11,2017-02-24,13.0,5622.23,4281.98,Approved,NaN
4,CLM-000005,BIL-000005,PAT-00566,Humana,2019-12-17,2020-01-02,16.0,3560.19,3174.32,Approved,NaN


In [43]:
insurance['year'] = insurance['claim_submitted_date'].dt.year
insurance['month'] = insurance['claim_submitted_date'].dt.month

In [44]:
# Check claim status distribution
print(insurance['claim_status'].value_counts())
print()
print(insurance['claim_status'].value_counts(normalize=True).multiply(100).round(1))


claim_status
Approved        5774
Pending         3332
Denied          2038
Under review     455
Name: count, dtype: int64

claim_status
Approved        49.8
Pending         28.7
Denied          17.6
Under review     3.9
Name: proportion, dtype: float64


In [45]:
# Find approval rate by insurance company
insurance.groupby('insurer_name')['claim_status'].value_counts(normalize=True).unstack().fillna(0).multiply(100).round(1)

claim_status,Approved,Denied,Pending,Under review
insurer_name,,,,
Aetna,50.0,18.4,27.9,3.7
BlueCross BlueShield,50.5,18.9,27.6,3.0
Cigna,50.0,19.0,26.6,4.3
Humana,46.9,17.4,31.9,3.9
Medicaid State Plan,50.4,17.1,28.8,3.7
Medicare Federal,49.9,16.8,29.2,4.1
UnitedHealth,49.4,17.5,28.3,4.8


In [46]:
# Analyze denial reason by insurance company
denied = insurance[insurance['claim_status'] == 'Denied']

denied['denial_reason'].value_counts()

denial_reason
Out-of-network provider       303
Service not covered           300
Exceeded benefit limit        296
Pre-authorization required    294
Incomplete documentation      292
Patient not eligible          282
Duplicate claim               271
Name: count, dtype: int64

In [47]:
# YoY analysis of insurance claims
insurance.groupby('year').agg(
    total_claims = ('claim_id', 'count'),
    approved_claim = ('claim_status', lambda x: (x == 'Approved').sum()),
    denied_claim = ('claim_status', lambda x: (x == 'Denied').sum()),
    total_amount = ('claim_amount', 'sum')
).assign(approval_rate = lambda df: (df['approved_claim'] / df['total_claims'] * 100).round(1))

,total_claims,approved_claim,denied_claim,total_amount,approval_rate
year,,,,,
2017,984,592,140,4186603.39,60.2
2018,1134,669,150,4689783.56,59.0
2019,1275,703,194,5435601.35,55.1
2020,2380,895,538,15899967.70,37.6
2021,2114,826,455,12602144.83,39.1
2022,1655,893,266,8073813.29,54.0
2023,1136,668,168,4951229.74,58.8
2024,921,528,127,3846439.63,57.3


In [48]:
# Claim status by payment status count
merged = insurance.merge(billing[['billing_id','payment_status']], on='billing_id')
merged.groupby(['claim_status','payment_status']).size().unstack(fill_value=0)

payment_status,Paid,Pending,Written-Off
claim_status,,,
Approved,5774,0,0
Denied,1007,0,1031
Pending,755,2577,0
Under review,455,0,0


In [49]:
# Insurance denial rate by Insurance type (in percentage)
merged2 = insurance.merge(patients[['patient_id','insurance_type']], on='patient_id')
merged2.groupby('insurance_type')['claim_status'] \
    .value_counts(normalize=True).unstack().fillna(0).multiply(100).round(1)

claim_status,Approved,Denied,Pending,Under review
insurance_type,,,,
Medicaid,50.2,17.1,28.9,3.8
Medicare,50.1,16.7,29.1,4.0
Private,49.3,18.2,28.4,4.0
